In [1]:
import yaml
from jinja2 import Template
from langsmith import Client

In [ ]:
jinja_template = """
you are a shopping assistant that can answer questions about products in stock.

You will be given a question and a list of context. 

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as avaialable products.
- As an output you need to provide

* answer of the question based on provided context.
* list of the IDs of the chunks used to answer the question. Only return the ones that are used in the answer.
* short description (1-2 sentences) of the items based on the description provided in the context.

- the short description should have the name of the item.
- the answer to the question should contain detailed information about the products and returned with detailed specifications of the products in bullet points.

Context:
{{ preprocessed_context }}

Question: {{ question }}
"""

In [3]:
template = Template(jinja_template)

In [4]:
rendered_template = template.render(preprocessed_context="this is the context", question="What is in stock?")

In [6]:
print(rendered_template)  # rendered_template


you are a shopping assistant that can answer questions about products in stock.

You will be given a question and a list of context. 

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as avaialable products.
- As an output you need to provide

* answer of the question based on provided context.
* list of the IDs of the chunks used to answer the question. Only return the ones that are used in the answer.
* short description (1-2 sentences) of the items based on the description provided in the context.

- the short description should have the name of the item.
- the answer to the question should contain detailed information about the products and returned with detailed specifications of the products in bullet points.

Context:
this is the context

Question: What is in stock?


In [29]:
def build_prompt_jinja(preprocessed_context, question):
    prompt = """
    You are a shopping assistant that can answer questions about products in stock.

    You will be given a question and a list of context. 

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as avaialable products.
    - As an output you need to provide

    * answer of the question based on provided context.
    * list of the IDs of the chunks used to answer the question. Only return the ones that are used in the answer.
    * short description (1-2 sentences) of the items based on the description provided in the context.

    - the short description should have the name of the item.
    - the answer to the question should contain detailed information about the products and returned with detailed specifications of the products in bullet points.

    Context:
    {{ preprocessed_context }}

    Question: {{ question }}
"""
    template = Template(prompt)
    rendered_prompt = template.render(
        preprocessed_context=preprocessed_context, 
        question=question
        )
    return rendered_prompt

In [30]:
print(build_prompt_jinja("this is the context", "What is in stock?"))


    You are a shopping assistant that can answer questions about products in stock.

    You will be given a question and a list of context. 

    Instructions:
    - You need to answer the question based on the provided context only.
    - Never use word context and refer to it as avaialable products.
    - As an output you need to provide

    * answer of the question based on provided context.
    * list of the IDs of the chunks used to answer the question. Only return the ones that are used in the answer.
    * short description (1-2 sentences) of the items based on the description provided in the context.

    - the short description should have the name of the item.
    - the answer to the question should contain detailed information about the products and returned with detailed specifications of the products in bullet points.

    Context:
    this is the context

    Question: What is in stock?


In [34]:
def prompt_template_config(yaml_file, prompt_key):
    with open(yaml_file, 'r') as file:
        config = yaml.safe_load(file)
    template_content = config["prompts"].get(prompt_key)
    # if not template_content:
    #     raise ValueError(f"Prompt key '{prompt_key}' not found in the YAML file.")
    prompt_template = Template(template_content)
    return prompt_template

In [35]:
def build_prompt_jinja(preprocessed_context, question):
    
    template = prompt_template_config("prompts/retrieval_generation.yaml", "retrieval_generation")
    rendered_prompt = template.render(
        preprocessed_context=preprocessed_context,
        question=question
        )
    return rendered_prompt

In [37]:
print(build_prompt_jinja("this is the Context", "What is in Stock?"))


You are a shopping assistant that can answer questions about products in stock.

You will be given a question and a list of context. 

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as avaialable products.
- As an output you need to provide

* answer of the question based on provided context.
* list of the IDs of the chunks used to answer the question. Only return the ones that are used in the answer.
* short description (1-2 sentences) of the items based on the description provided in the context.

- the short description should have the name of the item.
- the answer to the question should contain detailed information about the products and returned with detailed specifications of the products in bullet points.

Context:
this is the Context

Question: What is in Stock?


### Prompt Registries

In [38]:
ls_client = Client()

In [39]:
ls_template = ls_client.pull_prompt("retrieval-generation")

In [40]:
ls_template

ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'retrieval-generation', 'lc_hub_commit_hash': 'afe8606f6592da354e96922e0cfdaa103457896724244aa1022e13605b32199b'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a shopping assistant that can answer questions about products in stock.\nYou will be given a question and a list of context. \nInstructions:\n- You need to answer the question based on the provided context only.\n- Never use word context and refer to it as avaialable products.\n- As an output you need to provide\n* answer of the question based on provided context.\n* list of the IDs of the chunks used to answer the question. Only return the ones that are used in the answer.\n* short description (1-2 sentences) of the items based on the description provided in the context.\n- the short description should have the name o

In [42]:
print(ls_template.messages[0].prompt.template)

You are a shopping assistant that can answer questions about products in stock.
You will be given a question and a list of context. 
Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as avaialable products.
- As an output you need to provide
* answer of the question based on provided context.
* list of the IDs of the chunks used to answer the question. Only return the ones that are used in the answer.
* short description (1-2 sentences) of the items based on the description provided in the context.
- the short description should have the name of the item.
- the answer to the question should contain detailed information about the products and returned with detailed specifications of the products in bullet points.
Context:
{{ preprocessed_context }}
Question: {{ question }}


In [43]:
def prompt_template_registry(prompt_name):
    template_content = ls_client.pull_prompt(prompt_name).messages[0].prompt.template
    prompt_template = Template(template_content)
    return prompt_template

In [45]:
print(prompt_template_registry("retrieval-generation").render(preprocessed_context=" - this is the context1 \n - this is the context2", question="What is in Stock?"))

You are a shopping assistant that can answer questions about products in stock.
You will be given a question and a list of context. 
Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as avaialable products.
- As an output you need to provide
* answer of the question based on provided context.
* list of the IDs of the chunks used to answer the question. Only return the ones that are used in the answer.
* short description (1-2 sentences) of the items based on the description provided in the context.
- the short description should have the name of the item.
- the answer to the question should contain detailed information about the products and returned with detailed specifications of the products in bullet points.
Context:
 - this is the context1 
 - this is the context2
Question: What is in Stock?
